# CSE 151B — Development notebook (strategy-diverse sampling)

Rewrite of `dev.ipynb` using **strategy-level prompt diversity + decoding diversity + majority vote with diversity-weighted tiebreak**. Replaces the previous verify-pass approach (which produced 0 useful verdicts in testing) and the weaker meta-nudge variants with structurally different *problem-solving strategies* to break systematic-bias failure modes.

**Pipeline per question:**
1. **Diverse generation** — 3 strategy variants × 2 samples each = 6 samples per question:
   - `baseline` (T=0.6) — pure `precision_xml_verify` prompt; anchor on the known-good distribution.
   - `structured` (T=0.8) — Polya scaffold: **Understand → Plan → Execute → Verify**. Forces explicit parsing of *what is asked* and *what form the answer takes* before any computation.
   - `dual_repr` (T=0.9) — solve twice using two different representations (algebraic/numerical, symbolic/geometric, direct/limiting); only commit if both derivations agree.
2. **Selection** — majority vote, **tiebreak by number of distinct strategies** that produced the answer. No verifier pass. Trace cleanliness (`</think>` present, not truncated, shorter) breaks remaining ties on which specific trace to emit.

**Why strategy diversity and not meta-nudges?** Meta-nudges like "restate the question" overlap heavily with the baseline prompt's own verify-before-box clause — empirically too weak to shift the answer distribution. Strategy variants change the *reasoning structure itself* (forced phases, dual derivations), which is the only mechanism that has a chance of breaking systematic-bias errors like id=111 (5/6 samples wrote reciprocal-wrong answer under pure SC).

**Token budget:** `THINK_MAX_TOKENS=12288` per sample. Total per item ≈ 6×12k = 72k generated tokens. `dual_repr` will be the most token-heavy variant; budget tuning may be needed if it truncates often.

**Outputs:** `results/dev_results_pdiv_{N}_{Tk}k[_smoke].jsonl` plus:
- `.samples.jsonl` — all N raw samples per id with `prompt_tag` recorded
- `.responses.jsonl` — final per-id chosen trace


## 1. Environment

Colab (A100): run the `%pip` cell once, restart, continue. Local venv: install the same packages.


In [1]:
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 48.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 59.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 137.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 103.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 220.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 157.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 104.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━

## 2. Imports & configuration

| Knob | Meaning |
|------|---------|
| `THINK_MAX_TOKENS` | per-sample generation cap (default 12288 — leaves headroom under 16k total budget) |
| `PROMPT_VARIANTS` | list of `(tag, nudge_text_or_None, temperature)`; N=`N_PER_VARIANT × len(PROMPT_VARIANTS)` |
| `N_PER_VARIANT` | samples per variant (issued as one `SamplingParams(n=N_PER_VARIANT)` call per variant) |
| `SMOKE_TEST` / `SMOKE_N` | small-N debugging run on multi-blank FF only |


In [33]:
import json
import os
import re
import sys
import contextlib
import io
import glob
import site
from pathlib import Path
from typing import Optional


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    if (here.parent / "data").is_dir():
        return here.parent
    return here


REPO_ROOT = _repo_root()
DATA_PATH = None  # set in §4

# ── Decoding / sampling ───────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID   = "0"
THINK_MAX_TOKENS = 16368   # per-sample

# Prompt variants: each tuple is (tag, nudge_text_or_None, temperature).
# `nudge_text` is prepended to the user message inside <statement>.
# Each variant contributes N_PER_VARIANT samples via SamplingParams(n=N_PER_VARIANT).
PROMPT_VARIANTS: list[tuple[str, Optional[str], float]] = [
    ("baseline", None, 0.6),
    ("structured",
     "Solve this problem using the following structured approach:\n"
     "1. UNDERSTAND: Identify exactly what is given, what is asked, and the form/unit the answer must take. "
     "Flag any wording that could be misinterpreted (e.g., \'factor\' vs \'multiplier\', \'rate of change\' vs \'value\', "
     "\'compression by k\' vs \'compression to 1/k\', \'percentage\' vs \'probability\').\n"
     "2. PLAN: Name the theorem, formula, or technique that applies and briefly justify why it fits.\n"
     "3. EXECUTE: Apply the plan step-by-step, showing intermediate values.\n"
     "4. VERIFY: Test your answer — substitute back, check units/sign/magnitude, or use a limiting case. "
     "If the check fails, return to step 2 before boxing.",
     0.8),
    ("dual_repr",
     "Solve this problem TWICE using two genuinely different representations or methods. Pick whichever pair fits: "
     "algebraic vs. numerical, symbolic vs. geometric, direct computation vs. limiting case, formula vs. small-example "
     "generalization. State each derivation explicitly under a brief heading. Then compare the two results: if they "
     "agree, box the agreed answer; if they disagree, identify which derivation contains the error and box the corrected "
     "answer with a one-line note on which method was wrong and why.",
     0.9),
]
N_PER_VARIANT = 1
N_TOTAL_SAMPLES = N_PER_VARIANT * len(PROMPT_VARIANTS)

# Shared decoding knobs (only temp varies across variants)
TOP_P  = 0.95
TOP_K  = 40
MIN_P  = 0.0

# ── Smoke / run id ───────────────────────────────────────────────────────────
SMOKE_TEST = True
SMOKE_N    = 12

_TK = THINK_MAX_TOKENS // 1024
# Prompt build defined in §6 (system prompt). Baked into RUN_ID + OUTPUT_PATH so a
# prompt change cannot silently resume from a checkpoint written under an older prompt.
PROMPT_TAG = "exact_v1"
RUN_ID = f"dev-{PROMPT_TAG}-pdiv{N_TOTAL_SAMPLES}-{_TK}k" + ("-smoke" if SMOKE_TEST else "")
OUTPUT_PATH = str(REPO_ROOT / f"results/dev_results_{PROMPT_TAG}_pdiv{N_TOTAL_SAMPLES}_{_TK}k{'_smoke' if SMOKE_TEST else ''}.jsonl")

print(f"REPO_ROOT       = {REPO_ROOT}")
print(f"RUN_ID          = {RUN_ID}")
print(f"PROMPT_VARIANTS = {[(t, temp) for t,_,temp in PROMPT_VARIANTS]}")
print(f"N_PER_VARIANT   = {N_PER_VARIANT}   →  N_TOTAL_SAMPLES = {N_TOTAL_SAMPLES}")
print(f"THINK_MAX_TOKENS= {THINK_MAX_TOKENS}")
print(f"OUTPUT_PATH     = {OUTPUT_PATH}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"


def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs, seen = [], set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d); libdirs.append(d)
    if libdirs:
        os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(os.pathsep)


_prepend_nvidia_libs_to_ld_path()


@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close(); sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield


from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm


REPO_ROOT       = /content
RUN_ID          = dev-pdiv3-15k-smoke
PROMPT_VARIANTS = [('baseline', 0.6), ('structured', 0.8), ('dual_repr', 0.9)]
N_PER_VARIANT   = 1   →  N_TOTAL_SAMPLES = 3
THINK_MAX_TOKENS= 16368
OUTPUT_PATH     = /content/results/dev_results_pdiv_3_15k_smoke.jsonl


## 3. Colab: mount Drive (skip locally)

In [34]:
try:
    from google.colab import drive
except ImportError:
    print("Skip: not Colab.")
    DRIVE_PROJECT_ROOT = None
else:
    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    print(f"DRIVE_PROJECT_ROOT = {DRIVE_PROJECT_ROOT}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_PROJECT_ROOT = /content/drive/MyDrive/CSE151B


## 4. Copy frozen eval from Drive

Prefers `data/eval/holdout_20p.jsonl`. Falls back to a local copy if present.


In [35]:
import shutil

_EVAL_CANDIDATES = ("data/eval/holdout_20p.jsonl",)


def _resolve_eval_path() -> Path:
    for rel in _EVAL_CANDIDATES:
        local = REPO_ROOT / rel
        try:
            drive_root = DRIVE_PROJECT_ROOT
        except NameError:
            drive_root = None
        if drive_root is not None:
            src = drive_root / rel
            if src.is_file():
                local.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, local)
                print(f"Copied from Drive: {src} → {local.resolve()}")
                return local.resolve()
        if local.is_file():
            print(f"Using local eval: {local.resolve()}")
            return local.resolve()
    raise FileNotFoundError("No frozen eval JSONL found.")


DATA_PATH = str(_resolve_eval_path())


Copied from Drive: /content/drive/MyDrive/CSE151B/data/eval/holdout_20p.jsonl → /content/data/eval/holdout_20p.jsonl


## 5. Load dataset

In [36]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


data = [json.loads(line) for line in open(DATA_PATH)]

if SMOKE_TEST:
    multi = sorted(
        (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
        key=lambda d: d.get("id", 0),
    )
    data = multi[:SMOKE_N]
    print(f"SMOKE: {len(data)} multi-blank FF (ids {[d['id'] for d in data]})")

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Loaded {len(data)} questions ({n_mcq} MCQ, {n_free} free-form) from {DATA_PATH}")


SMOKE: 12 multi-blank FF (ids [6, 20, 21, 44, 46, 65, 100, 105, 111, 127, 128, 139])
Loaded 12 questions (0 MCQ, 12 free-form) from /content/data/eval/holdout_20p.jsonl


## 6. Prompt construction (`exact_v1` + style nudges)

`exact_v1` system prompt (see code cell), the post-`priv-002` rewrite of `precision_xml_verify`:

1. **Precision** — prefer exact closed forms; decimals only when unavoidable, at conventional precision, **never digit-padded** (the grader's 1e-8 relative tolerance rejects over-precise decimals like `1.6448536270` against rounded gold `1.645`).
2. **No representation gamble** — simplest standard form; echo the question's own True/False / Yes/No vocabulary.
3. **Plain text** — drops the XML `<rule>`/`<instructions>` scaffolding (24% rule-echo) and the verify clause; keeps only a light "be concise" directive.

**Multi-blank** now emits a **single** `\boxed{a, b, c}` box (grader-identical to separate boxes via bracket-aware `split_by_comma`, and immune to the contiguous-group / separator pitfalls of multiple boxes).

Style nudges prepend a single `Approach:` sentence to the user message to bias the reasoning path without changing the format contract.


In [ ]:
# ── Prompt variant: exact_v1 ──────────────────────────────────────────────────
# Rewrite of precision_xml_verify after the priv-002 trace analysis
# (docs/analysis/private-submission-precision-xml-verify-priv-002.md). Three fixes:
#   (1) PRECISION  — exact forms preferred; decimals only when unavoidable, at
#                    conventional precision, NEVER digit-padded. The grader's 1e-8
#                    relative tolerance rejects over-precise decimals vs rounded gold
#                    (e.g. 1.6448536270 fails against gold 1.645).
#   (2) NO REP GAMBLE — keep the simplest standard form; echo the question's own
#                    True/False / Yes/No vocabulary; don't rewrite into equivalents.
#   (3) PLAIN TEXT — drop the XML <rule>/<instructions> scaffolding (caused 24% rule
#                    echo) and the verify clause (adds length, flips answers). Keep
#                    only a light "be concise" directive (recovered ~26 truncations).
# Multi-blank now emits ONE box \boxed{a, b, c} (grader-identical to separate boxes
# via bracket-aware split_by_comma, and immune to the contiguous-group/separator
# pitfalls of multiple boxes).

_ROLE_SOLVE = (
    "You are a precise mathematical reasoner. Solve the problem rigorously, but keep "
    "your reasoning concise: do not re-derive steps you have already completed or go "
    "in circles."
)
_ROLE_MCQ = (
    "You are a precise mathematical reasoner. Read the problem and the answer choices, "
    "then identify the single best answer. Keep your reasoning concise; you may confirm "
    "by elimination."
)

_PRECISION_RULE = (
    "Answer precision is graded against a rounded reference with a very tight tolerance, "
    "so the FORM of your answer matters:\n"
    "- Prefer an EXACT closed form whenever one exists - fractions (5/8), radicals "
    "(3\\sqrt{2}), \\pi, e, logarithms (\\log_3 35). Box it exactly; the grader matches "
    "exact forms reliably and they carry no rounding error.\n"
    "- Use a decimal ONLY when no exact form exists (e.g. a statistical test statistic, "
    "p-value, regression coefficient, or physical measurement). Then use the CONVENTIONAL "
    "precision for the problem: standard statistical constants in their usual rounded form "
    "(z = 1.96, 1.645, 2.576; the usual t-table values), and otherwise 2-4 decimal places, "
    "or the precision of the data you were given.\n"
    "- NEVER pad an answer with extra significant figures. An over-precise decimal such as "
    "1.6448536270 will FAIL against a rounded gold value like 1.645. Do not round a true "
    "exact form to a decimal, and do not over-expand a conventional decimal."
)

_REPRESENTATION_RULE = (
    "Give the answer in the simplest standard form and do not rewrite it into a fancier "
    "equivalent (keep \\log_3 35 rather than \\frac{\\ln 35}{\\ln 3}; keep a clean factored "
    "or simplified expression). For True/False or Yes/No answers, use exactly the words the "
    "question uses - write True / False when it says 'True or False', and Yes / No when it "
    "asks a yes/no question."
)

_FORMAT_RULE = (
    "Box the value only - no units or surrounding words (\\boxed{5}, not \\boxed{5 cm}). "
    "Write a probability as a decimal (0.5, not 50%) and a percentage as a bare number "
    "(50, no % sign). Keep any coordinate, tuple, interval, or set together with its "
    "brackets, e.g. \\boxed{(-1, -3)} or \\boxed{[0, 14]}, so its internal commas are not "
    "mistaken for answer separators."
)

# Final-answer layout: always exactly ONE \boxed{} on a single Final Answer line.
_LAYOUT_SINGLE = (
    "End with a single line 'Final Answer:' followed by exactly one \\boxed{} containing "
    "your answer. Put \\boxed{} nowhere else in the response."
)
_LAYOUT_MCQ = (
    "End with a single line 'Final Answer:' followed by exactly one \\boxed{} containing "
    "ONLY the letter of your chosen option (e.g. \\boxed{C}). If the slot says 'select all "
    "that apply', concatenate the letters alphabetically inside that one box with no "
    "separator (e.g. \\boxed{AB}). Put \\boxed{} nowhere else in the response."
)


def _math_layout_rule(n_blanks: int) -> str:
    if n_blanks > 1:
        return (
            f"This problem has {n_blanks} blanks ([ANS]). End with a single line "
            f"'Final Answer:' followed by exactly ONE \\boxed{{}} that contains all "
            f"{n_blanks} answers, comma-separated, in the order the [ANS] blanks appear, "
            f"e.g. \\boxed{{a, b, c}}. Keep any tuple/interval/set in its own brackets "
            f"inside the box. Use only this one box; put \\boxed{{}} nowhere else."
        )
    return _LAYOUT_SINGLE


SYSTEM_MATH = "\n\n".join([_ROLE_SOLVE, _PRECISION_RULE, _REPRESENTATION_RULE, _FORMAT_RULE])
SYSTEM_MCQ  = "\n\n".join([_ROLE_MCQ, _REPRESENTATION_RULE])


def _build_user(question: str, options, nudge: Optional[str] = None) -> str:
    body = question if not nudge else f"{nudge}\n\n{question}"
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return f"{body}\n\nOptions:\n{opts}\n\n{_LAYOUT_MCQ}"
    n_blanks = n_ans_placeholders(question)
    return f"{body}\n\n{_math_layout_rule(n_blanks)}"


def build_prompt(question: str, options, nudge: Optional[str] = None) -> tuple[str, str]:
    system = SYSTEM_MCQ if options else SYSTEM_MATH
    return system, _build_user(question, options, nudge=nudge)


# Demo
_demo_multi = next((d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1), None)
_demo = _demo_multi or data[0]
sys_p, usr_p = build_prompt(_demo["question"], _demo.get("options"))
print("-- SYSTEM_MATH --")
print(sys_p)
print("\n-- user (multi-blank demo, first 500) --")
print(usr_p[:500], "...")
_mcq = next((d for d in data if d.get("options")), None)
if _mcq:
    s2, u2 = build_prompt(_mcq["question"], _mcq.get("options"))
    print("\n-- user (MCQ demo tail) --")
    print(u2[-400:])


## 7. Load vLLM (A100 bf16, FP8 KV cache)

Matches the existing dev profile. `max_model_len` sized for prompt + `THINK_MAX_TOKENS` + verify pass.


In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

with _jupyter_stdout_for_vllm():
    llm = LLM(
        model=MODEL_ID,
        dtype="bfloat16",
        enable_prefix_caching=True,
        gpu_memory_utilization=0.92,
        max_model_len=17500,
        trust_remote_code=True,
        max_num_seqs=256,
        max_num_batched_tokens=32768,
        enable_chunked_prefill=True,
        kv_cache_dtype="fp8",
    )

THINKING_OPEN  = "<think>"
THINKING_CLOSE = "</think>"

print("Model loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.8k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

INFO 06-01 01:24:37 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=17500, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=fp8, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, co

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

INFO 06-01 01:24:58 [weight_utils.py:615] Time spent downloading weights for Qwen/Qwen3-4B-Thinking-2507: 19.246449 seconds
INFO 06-01 01:24:59 [weight_utils.py:904] Filesystem type for checkpoints: OVERLAY. Checkpoint size: 7.49 GiB. Available RAM: 79.84 GiB.
INFO 06-01 01:24:59 [weight_utils.py:927] Auto-prefetch is disabled because the filesystem (OVERLAY) is not a recognized network FS (NFS/Lustre). If you want to force prefetching, start vLLM with --safetensors-load-strategy=prefetch.


Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 06-01 01:25:01 [default_loader.py:384] Loading weights took 2.38 seconds
INFO 06-01 01:25:02 [gpu_model_runner.py:4879] Model loading took 7.61 GiB memory and 22.654167 seconds
INFO 06-01 01:25:13 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/d5f944a7c1/rank_0_0/backbone for vLLM's torch.compile
INFO 06-01 01:25:13 [backends.py:1128] Dynamo bytecode transform time: 10.58 s
INFO 06-01 01:25:23 [backends.py:376] Cache the graph of compile range (1, 32768) for later use
INFO 06-01 01:25:29 [backends.py:391] Compiling a graph for compile range (1, 32768) takes 15.62 s
INFO 06-01 01:25:36 [decorators.py:668] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/d274bfa70d8d3efd0218761bc4a7f48ef22308cc6339102c69338a4e0db39b80/rank_0_0/model
INFO 06-01 01:25:36 [monitor.py:53] torch.compile took 33.39 s in total
INFO 06-01 01:25:36 [monitor.py:81] Initial profiling/warmup run took 0.19 s
INFO 06-01 01:25:47 [gpu_model_run

## 8. Answer extraction & canonicalization

Re-uses the dev-SC canon helpers: extract `\boxed{}` group via `judger.extract_all_boxed`, split multi-value boxes on top-level commas, canonicalize each (latex → sympy → numeric round). Two traces are considered the same answer iff their canonical tuples match.


In [38]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

import sympy
from collections import Counter
from sympy.parsing.latex import parse_latex

CANON_NUMERIC_PREC = 6
_SET_INTERVAL_RE = re.compile(r"^[\{\[\(].*[\}\]\)]$")


def _strip_units(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\\,.*$", "", s).strip()
    s = re.sub(r"\s*\\text\{[^}]*\}\s*$", "", s).strip()
    return s


def _split_multi_value_box(s: str) -> list[str]:
    s = s.strip()
    if not s or _SET_INTERVAL_RE.match(s):
        return [s]
    if "," not in s:
        return [s]
    parts, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch == "{": depth += 1
        elif ch == "}": depth = max(0, depth - 1)
        elif ch == "," and depth == 0:
            parts.append(s[start:i].strip()); start = i + 1
    parts.append(s[start:].strip())
    return [p for p in parts if p] if len(parts) > 1 else [s]


def _canon(s: str) -> str:
    s = _strip_units(s)
    if not s:
        return ""
    if _SET_INTERVAL_RE.match(s):
        return s.replace(" ", "").lower()
    for parser in (parse_latex, sympy.sympify):
        try:
            expr = parser(s)
            if expr.is_number or getattr(expr, "is_Number", False):
                f = float(expr.evalf())
                return format(round(f, CANON_NUMERIC_PREC), f".{CANON_NUMERIC_PREC}g")
            return str(sympy.simplify(expr))
        except Exception:
            pass
    s = s.replace(" ", "").replace("\\frac", "/").replace("\\sqrt", "sqrt")
    s = s.replace("{", "").replace("}", "")
    return s.lower()


def _canon_box_list(boxes: list[str]) -> list[str]:
    out = []
    for b in boxes:
        for part in _split_multi_value_box(b):
            out.append(_canon(part))
    return out


def _extract_boxes(text: str) -> list[str]:
    if THINKING_CLOSE not in text:
        return []
    try:
        return judger.extract_all_boxed(text) or []
    except Exception:
        return []


def _extract_mcq_letter(text: str) -> Optional[str]:
    if THINKING_CLOSE not in text:
        return None
    m = re.search(r"\\boxed\{([A-Za-z]+)\}", text)
    return m.group(1).upper() if m else None


def canonical_answer(trace: str, item: dict) -> Optional[tuple]:
    """Return a hashable canonical-answer tuple for vote grouping, or None if no parsable answer."""
    if item.get("options"):
        let = _extract_mcq_letter(trace)
        return (let,) if let else None
    n_blanks = n_ans_placeholders(item["question"])
    boxes = _extract_boxes(trace)
    if not boxes:
        return None
    if n_blanks <= 1:
        return tuple(_canon_box_list([boxes[-1]])) or None
    if len(boxes) != n_blanks:
        return None
    canon = _canon_box_list(boxes)
    return tuple(canon) if len(canon) == n_blanks else None


def display_answer(trace: str, item: dict) -> str:
    """Human-readable string of the trace's answer for verify prompts."""
    if item.get("options"):
        return _extract_mcq_letter(trace) or ""
    boxes = _extract_boxes(trace)
    if not boxes:
        return ""
    n_blanks = n_ans_placeholders(item["question"])
    if n_blanks <= 1:
        return boxes[-1]
    return ", ".join(boxes[:n_blanks])


print("Canon helpers loaded.")


JUDGER_ROOT=/content/drive/MyDrive/CSE151B
Canon helpers loaded.


## 9. Diverse generation (strategy × decoding)

One vLLM call per strategy in `PROMPT_VARIANTS`, each issuing `SamplingParams(n=N_PER_VARIANT)` at that strategy's temperature. Strategies differ in `(nudge_text, temperature)` — same system prompt, different user-prompt prefix and decoding stochasticity.

The 3 strategies (`baseline`, `structured`, `dual_repr`) are deliberately *structurally* different: same problem, different reasoning scaffold. The within-strategy `n=N_PER_VARIANT` samples give stochasticity inside the same scaffold, so the selection-time diversity weighting can distinguish "strategy agreed with itself" from "strategy was noisy".

Outputs per id: `{"id", "samples": [{"tag", "prompt_tag", "text", "finish_reason"}], ...}` where `prompt_tag` is the strategy tag used by selection (§11).


In [39]:
def _build_chat_text(system: str, user: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True,
    )


def _variant_sp(temp: float) -> SamplingParams:
    return SamplingParams(
        n=N_PER_VARIANT,
        max_tokens=THINK_MAX_TOKENS,
        temperature=temp,
        top_p=TOP_P,
        top_k=TOP_K,
        min_p=MIN_P,
    )


def generate_diverse(items: list[dict]) -> dict[int, list[dict]]:
    """Return {id: [sample_dict, ...]} of length N_TOTAL_SAMPLES per item.

    sample_dict = {"tag", "prompt_tag", "text", "finish_reason"}.
    """
    out: dict[int, list[dict]] = {it["id"]: [] for it in items}

    for variant_tag, nudge, temp in PROMPT_VARIANTS:
        prompts = []
        for it in items:
            sys_p, usr_p = build_prompt(it["question"], it.get("options"), nudge=nudge)
            prompts.append(_build_chat_text(sys_p, usr_p))
        with _jupyter_stdout_for_vllm():
            ros = llm.generate(prompts, sampling_params=_variant_sp(temp))
        for it, ro in zip(items, ros):
            for j, o in enumerate(ro.outputs):
                out[it["id"]].append({
                    "tag": f"{variant_tag}_s{j}",
                    "prompt_tag": variant_tag,
                    "text": o.text.strip(),
                    "finish_reason": o.finish_reason,
                })

    return out


print(f"Diverse-gen ready: {len(PROMPT_VARIANTS)} variants × {N_PER_VARIANT} samples = {N_TOTAL_SAMPLES}/item.")
print(f"  variants: {[(t, temp) for t,_,temp in PROMPT_VARIANTS]}")


Diverse-gen ready: 3 variants × 1 samples = 3/item.
  variants: [('baseline', 0.6), ('structured', 0.8), ('dual_repr', 0.9)]


## 11. Selection — majority vote + diversity-weighted tiebreak

For each item:
1. Group samples by canonical answer.
2. Score each canonical answer = `vote_count + 0.5 × n_distinct_prompt_variants`. The diversity term breaks ties in favor of answers supported by *different framings* — independent evidence is stronger than repeated evidence from the same prompt.
3. Pick the cleanest trace in the winning group (`finish_reason != "length"`, has `</think>`, shorter trace).
4. If no sample is parseable, fall back to the cleanest trace overall.


In [40]:
def _trace_cleanliness(sample: dict) -> tuple:
    """Sort key: lower = cleaner. Prefer finished + has </think> + shorter text."""
    finished_ok = 0 if sample["finish_reason"] != "length" else 1
    has_close   = 0 if THINKING_CLOSE in sample["text"] else 1
    return (finished_ok, has_close, len(sample["text"]))


def select_response(item: dict, samples: list[dict]) -> dict:
    """Returns {"response": str, "picked_idx": int, "picked_canon": tuple|None, "stats": {...}}."""
    groups: dict[Optional[tuple], list[int]] = {}
    for idx, s in enumerate(samples):
        c = canonical_answer(s["text"], item)
        groups.setdefault(c, []).append(idx)

    scored: list[tuple[float, tuple, list[int], int]] = []
    for canon, idxs in groups.items():
        if canon is None:
            continue
        n_distinct_prompts = len({samples[i]["prompt_tag"] for i in idxs})
        score = len(idxs) + 0.5 * n_distinct_prompts
        scored.append((score, canon, idxs, n_distinct_prompts))

    if scored:
        # Sort by (-score, -vote_count, -distinct_prompts) so ties prefer wider support.
        scored.sort(key=lambda t: (-t[0], -len(t[2]), -t[3]))
        best_score, best_canon, best_idxs, best_n_prompts = scored[0]
        best_idxs.sort(key=lambda i: _trace_cleanliness(samples[i]))
        picked = best_idxs[0]
        return {
            "response": samples[picked]["text"],
            "picked_idx": picked,
            "picked_canon": best_canon,
            "stats": {
                "n_unique": len(scored),
                "winning_votes": len(best_idxs),
                "winning_n_prompts": best_n_prompts,
                "winning_score": best_score,
                "n_unparseable": len(groups.get(None, [])),
            },
        }

    fallback = sorted(range(len(samples)), key=lambda i: _trace_cleanliness(samples[i]))[0]
    return {
        "response": samples[fallback]["text"],
        "picked_idx": fallback,
        "picked_canon": None,
        "stats": {"n_unique": 0, "winning_votes": 0, "winning_n_prompts": 0,
                  "winning_score": 0.0, "n_unparseable": len(samples)},
    }


print("Selection helpers loaded (majority vote + diversity-weighted tiebreak).")


Selection helpers loaded (majority vote + diversity-weighted tiebreak).


## 12. Run pipeline (generate → select)

Checkpoints:
- `<output>.samples.jsonl` — all N raw samples per id (skip generation on resume).
- `<output>.responses.jsonl` — final per-id chosen trace.

Delete a checkpoint to force re-do of that stage.


In [41]:
CHUNK_SIZE = max(8, len(data) // 4)  # 4 generation chunks by default

try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(OUTPUT_PATH).parent
_results_dir.mkdir(parents=True, exist_ok=True)

samples_ckpt   = _results_dir / (Path(OUTPUT_PATH).stem + ".samples.jsonl")
response_ckpt  = _results_dir / (Path(OUTPUT_PATH).stem + ".responses.jsonl")
print(f"samples checkpoint  : {samples_ckpt}")
print(f"response checkpoint : {response_ckpt}")

samples_by_id: dict[int, list[dict]] = {}
if samples_ckpt.exists():
    with open(samples_ckpt) as f:
        for line in f:
            r = json.loads(line)
            samples_by_id[int(r["id"])] = r["samples"]
    print(f"Resumed samples: {len(samples_by_id)} ids")

remaining = [d for d in data if d["id"] not in samples_by_id]
print(f"Generating samples for {len(remaining)} / {len(data)} items (chunk={CHUNK_SIZE})")

by_id = {d["id"]: d for d in data}

for cs in range(0, len(remaining), CHUNK_SIZE):
    chunk = remaining[cs : cs + CHUNK_SIZE]
    chunk_samples = generate_diverse(chunk)
    with open(samples_ckpt, "a") as f:
        for it in chunk:
            samples_by_id[it["id"]] = chunk_samples[it["id"]]
            f.write(json.dumps({"id": it["id"], "samples": chunk_samples[it["id"]]}) + "\n")
    print(f"  samples: {len(samples_by_id)}/{len(data)}")


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

samples checkpoint  : /content/drive/MyDrive/CSE151B/results/dev_results_pdiv_3_15k_smoke.samples.jsonl
response checkpoint : /content/drive/MyDrive/CSE151B/results/dev_results_pdiv_3_15k_smoke.responses.jsonl
Generating samples for 12 / 12 items (chunk=8)


Rendering prompts:   0%|          | 0/8 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/8 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  samples: 12/12


In [42]:
# ── Stage C: select final response per id ─────────────────────────────────────
completed: dict[int, str] = {}
selection_stats: list[dict] = []
for qid, samples in samples_by_id.items():
    item = by_id[qid]
    sel = select_response(item, samples)
    completed[qid] = sel["response"]
    selection_stats.append({"id": qid, **sel["stats"], "picked_idx": sel["picked_idx"]})

with open(response_ckpt, "w") as f:
    for d in data:
        f.write(json.dumps({"id": d["id"], "response": completed[d["id"]]}) + "\n")
print(f"Selected & saved {len(completed)} responses → {response_ckpt.name}")

if selection_stats:
    n = len(selection_stats)
    avg_unique  = sum(s["n_unique"] for s in selection_stats) / n
    avg_unparse = sum(s["n_unparseable"] for s in selection_stats) / n
    avg_winvote = sum(s["winning_votes"] for s in selection_stats) / n
    avg_winprmpt= sum(s["winning_n_prompts"] for s in selection_stats) / n
    n_consensus = sum(1 for s in selection_stats if s["winning_n_prompts"] == len(PROMPT_VARIANTS))
    print(f"Selection summary ({n} items, N={N_TOTAL_SAMPLES}):")
    print(f"  avg unique canon answers / item : {avg_unique:.2f}")
    print(f"  avg unparseable samples / item  : {avg_unparse:.2f} / {N_TOTAL_SAMPLES}")
    print(f"  avg winning vote count          : {avg_winvote:.2f} / {N_TOTAL_SAMPLES}")
    print(f"  avg distinct prompts in winner  : {avg_winprmpt:.2f} / {len(PROMPT_VARIANTS)}")
    print(f"  cross-prompt consensus items    : {n_consensus}/{n} ({100*n_consensus/n:.1f}%)  (winner backed by ALL variants)")


Selected & saved 12 responses → dev_results_pdiv_3_15k_smoke.responses.jsonl
Selection summary (12 items, N=3):
  avg unique canon answers / item : 1.42
  avg unparseable samples / item  : 0.08 / 3
  avg winning vote count          : 2.50 / 3
  avg distinct prompts in winner  : 2.50 / 3
  cross-prompt consensus items    : 8/12 (66.7%)  (winner backed by ALL variants)


## 13. Score responses

In [43]:
responses = [completed[d["id"]] for d in data]


def score_mcq(response: str, gold_letter: str) -> bool:
    extracted = (judger.extract_ans(response) or "").strip().upper()
    gold = str(gold_letter).strip().upper()
    if len(gold) > 1:
        return judger.judge_MC_multiple(extracted, gold)
    return judger.judge_MC_single(extracted, gold)


results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        except Exception:
            correct = False
    results.append({
        "id": item.get("id"), "is_mcq": is_mcq, "gold": gold,
        "response": response, "correct": correct,
    })

print(f"Scored {len(results)} items.")


Scoring: 100%|██████████| 12/12 [00:01<00:00,  6.99it/s]

Scored 12 items.


## 14. Summary

In [44]:
from collections import Counter

mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]
_q_by_id = {d["id"]: d["question"] for d in data}
multi_free  = [r for r in free_res if n_ans_placeholders(_q_by_id[r["id"]]) > 1]
single_free = [r for r in free_res if r not in multi_free]


def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0


print("=" * 56)
print("DEV RESULTS — prompt-diverse sampling + diversity-weighted vote")
print("=" * 56)
print(f"  RUN_ID            : {RUN_ID}")
print(f"  PROMPT_VARIANTS   : {[(t,temp) for t,_,temp in PROMPT_VARIANTS]}")
print(f"  N_PER_VARIANT     : {N_PER_VARIANT}  →  N_TOTAL_SAMPLES = {N_TOTAL_SAMPLES}")
print(f"  THINK_MAX_TOKENS  : {THINK_MAX_TOKENS}")
print(f"  SMOKE             : {SMOKE_TEST}")
print(f"  N                 : {len(results)}  ({len(mcq_res)} MCQ, {len(free_res)} FF)")
print("-" * 56)
if mcq_res:
    print(f"  MCQ          : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
if free_res:
    print(f"  Free-form    : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
if multi_free:
    print(f"  Multi-blank  : {sum(r['correct'] for r in multi_free):4d} / {len(multi_free):4d}  ({acc(multi_free):.2f}%)")
if single_free:
    print(f"  Single-blank : {sum(r['correct'] for r in single_free):4d} / {len(single_free):4d}  ({acc(single_free):.2f}%)")
print(f"  Overall      : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 56)

# Pick rate by prompt variant — diagnostic for whether non-baseline variants contribute.
# If baseline gets picked ~100% of the time, the variants aren't doing work.
prompt_picked = Counter()
for s in selection_stats:
    picked_sample = samples_by_id[s["id"]][s["picked_idx"]]
    prompt_picked[picked_sample["prompt_tag"]] += 1
print("\nPick rate by prompt variant (which framing's trace won selection):")
for tag, _, temp in PROMPT_VARIANTS:
    pct = 100 * prompt_picked[tag] / len(selection_stats) if selection_stats else 0
    print(f"  {tag:<12} (T={temp}):  picked {prompt_picked[tag]:>3} / {len(selection_stats)}  ({pct:.1f}%)")


DEV RESULTS — prompt-diverse sampling + diversity-weighted vote
  RUN_ID            : dev-pdiv3-15k-smoke
  PROMPT_VARIANTS   : [('baseline', 0.6), ('structured', 0.8), ('dual_repr', 0.9)]
  N_PER_VARIANT     : 1  →  N_TOTAL_SAMPLES = 3
  THINK_MAX_TOKENS  : 16368
  SMOKE             : True
  N                 : 12  (0 MCQ, 12 FF)
--------------------------------------------------------
  Free-form    :    5 /   12  (41.67%)
  Multi-blank  :    5 /   12  (41.67%)
  Overall      :    5 /   12  (41.67%)

Pick rate by prompt variant (which framing's trace won selection):
  baseline     (T=0.6):  picked   7 / 12  (58.3%)
  structured   (T=0.8):  picked   4 / 12  (33.3%)
  dual_repr    (T=0.9):  picked   1 / 12  (8.3%)


## 15. Save results (`{id, is_mcq, gold, response, correct}` schema)

In [45]:
SAVE_EVAL = True

try:
    out_path = DRIVE_PROJECT_ROOT / "results" / Path(OUTPUT_PATH).name
except NameError:
    out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            rec = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                   "response": r["response"], "correct": r["correct"]}
        else:
            rec = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(rec) + "\n")

print(f"Saved {len(results)} records → {out_path.resolve()}")


Saved 12 records → /content/drive/MyDrive/CSE151B/results/dev_results_pdiv_3_15k_smoke.jsonl


In [46]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not Colab — skipping runtime termination.")
